In [0]:
from pyspark.sql.types import *
from pyspark.sql.functions import *

In [0]:
bronze_products_df = spark.table(
    "ecommerce_lakehouse.bronze.products_raw"
)

In [0]:
silver_products_df = bronze_products_df \
    .dropDuplicates(["product_id"]) \
    .withColumn(
        "product_category_name",
        initcap(col("product_category_name"))
    )

In [0]:
silver_products_df = silver_products_df \
    .withColumnRenamed("product_description_lenght","product_description_length") \
    .withColumnRenamed("product_name_lenght","product_name_length")


In [0]:
display(silver_products_df)

In [0]:
silver_products_df = silver_products_df.withColumn(
    "product_volume_cm3",
    col("product_length_cm")
    * col("product_height_cm")
    * col("product_width_cm")
)

In [0]:
silver_products_df = silver_products_df.withColumn(
    "weight_category",
    when(col("product_weight_g") < 500, "Light")
    .when(col("product_weight_g") < 2000, "Medium")
    .otherwise("Heavy")
)

In [0]:
(
    silver_products_df.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(
            "ecommerce_lakehouse.silver.products_clean"
        )
)

In [0]:
%sql
SELECT *
FROM ecommerce_lakehouse.silver.products_clean
LIMIT 10;